In [0]:
# Silver layer implementation

from pyspark.sql import functions as F

customers = spark.table(
    "retail_catalog.bronze.customers"
)

products = spark.table(
    "retail_catalog.bronze.products"
)

sales = spark.table(
    "retail_catalog.bronze.sales"
)

In [0]:
customers_clean = customers.select(
    F.expr("try_cast(CustomerID AS INT)")
        .alias("CustomerID"),

    F.initcap(F.trim("CustomerName"))
        .alias("CustomerName"),

    F.initcap(F.trim("City"))
        .alias("City"),

    F.expr("try_cast(RegistrationDate AS DATE)")
        .alias("RegistrationDate")
).filter(
    F.col("CustomerID").isNotNull()
).dropDuplicates(
    ["CustomerID"]
)

display(customers_clean)

In [0]:
products_clean = products.select(
    F.expr("try_cast(ProductID AS INT)")
        .alias("ProductID"),

    F.trim("ProductName")
        .alias("ProductName"),

    F.initcap(F.trim("Category"))
        .alias("Category"),

    F.expr("try_cast(UnitPrice AS DECIMAL(12,2))")
        .alias("UnitPrice")
).filter(
    F.col("ProductID").isNotNull() &
    (F.col("UnitPrice") > 0)
).dropDuplicates(
    ["ProductID"]
)

display(products_clean)

In [0]:
customers_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "retail_catalog.silver.customers"
    )

products_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "retail_catalog.silver.products"
    )

In [0]:
sales_typed = sales.select(

    F.expr("try_cast(SalesID AS INT)")
        .alias("SalesID"),

    F.expr("try_cast(CustomerID AS INT)")
        .alias("CustomerID"),

    F.expr("try_cast(ProductID AS INT)")
        .alias("ProductID"),

    F.expr("try_cast(Quantity AS INT)")
        .alias("Quantity"),

    F.col("SalesDate").alias("RawSalesDate"),

    F.expr("try_cast(SalesDate AS DATE)")
        .alias("SalesDate"),

    "IngestionTimestamp",
    "SourceFile"
)

display(sales_typed)

In [0]:
sales_deduplicated = sales_typed.dropDuplicates(
    [
        "SalesID",
        "CustomerID",
        "ProductID",
        "Quantity",
        "RawSalesDate"
    ]
)

duplicate_count = (
    sales_typed.count() -
    sales_deduplicated.count()
)

print("Duplicate records removed:", duplicate_count)

Duplicate records removed: 1


In [0]:
enriched_sales = sales_deduplicated.alias("s") \
    .join(
        customers_clean.alias("c"),
        F.col("s.CustomerID") ==
        F.col("c.CustomerID"),
        "left"
    ) \
    .join(
        products_clean.alias("p"),
        F.col("s.ProductID") ==
        F.col("p.ProductID"),
        "left"
    ) \
    .select(
        "s.*",
        F.col("c.CustomerID").alias("MatchedCustomerID"),
        F.col("c.CustomerName"),
        F.col("c.City"),
        F.col("p.ProductID").alias("MatchedProductID"),
        F.col("p.ProductName"),
        F.col("p.Category"),
        F.col("p.UnitPrice")
    )

display(enriched_sales)

In [0]:
validated_sales = enriched_sales.withColumn(

    "RejectionReason",

    F.when(
        F.col("SalesID").isNull(),
        "Missing Sales ID"

    ).when(
        F.col("CustomerID").isNull(),
        "Missing Customer ID"

    ).when(
        F.col("ProductID").isNull(),
        "Missing Product ID"

    ).when(
        F.col("Quantity").isNull(),
        "Missing Quantity"

    ).when(
        F.col("Quantity") <= 0,
        "Invalid Quantity"

    ).when(
        F.col("SalesDate").isNull(),
        "Invalid Sales Date"

    ).when(
        F.col("MatchedCustomerID").isNull(),
        "Unknown Customer"

    ).when(
        F.col("MatchedProductID").isNull(),
        "Unknown Product"

    ).otherwise(
        F.lit(None).cast("string")
    )
)

In [0]:
valid_sales = validated_sales.filter(
    F.col("RejectionReason").isNull()
)

rejected_sales = validated_sales.filter(
    F.col("RejectionReason").isNotNull()
)

print("Valid records:", valid_sales.count())

print("Rejected records:", rejected_sales.count())

Valid records: 14
Rejected records: 6


In [0]:
sales_clean = valid_sales.withColumn(
    "Revenue",
    (F.col("Quantity") * F.col("UnitPrice"))
        .cast("decimal(18,2)")
).withColumn(
    "SalesYear",
    F.year("SalesDate")
).withColumn(
    "SalesMonth",
    F.month("SalesDate")
).withColumn(
    "ProcessedTimestamp",
    F.current_timestamp()
).drop(
    "MatchedCustomerID",
    "MatchedProductID",
    "RejectionReason",
    "RawSalesDate"
)

display(sales_clean)

In [0]:
sales_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "retail_catalog.silver.sales_clean"
    )

rejected_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "retail_catalog.silver.sales_rejected"
    )

In [0]:
%sql
SELECT
    SalesID,
    CustomerID,
    ProductID,
    Quantity,
    RawSalesDate,
    RejectionReason
FROM retail_catalog.silver.sales_rejected
ORDER BY SalesID;

In [0]:
bronze_count = sales.count()

valid_count = sales_clean.count()

rejected_count = rejected_sales.count()

assert bronze_count == (
    valid_count +
    rejected_count +
    duplicate_count
), "Data reconciliation failed"

print("Data reconciliation successful")

Data reconciliation successful
